# Phase 5: PU-Learning & Random Forest 베이스 모델 프로토타이핑
이 노트북은 실제 데이터(API 상권 피처 및 공공 투기 라벨)를 생성하기 전, **가상의 더미(Dummy) 데이터** 행렬을 이용하여 논문에서 제안한 **PU-Bagging + RandomForest 구조**가 파이썬 코드로 문법 에러 없이 작동하는지 검증(Test)하기 위해 만들어진 프로토타입입니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("✅ 기본 라이브러리 및 Scikit-Learn 환경 로드 완료!")

## 1. 더미 데이터 생성 (Dummy Data Generation)
상권 데이터 분포를 빙자한 가상의 X(Feature), y(Label)를 `make_classification`을 통해 10,000개 정도 무작위로 찍어냅니다.

* **Positive (1):** 실제 쓰레기가 투기된 구역 (전체의 약 10%)
* **Unlabeled (0):** 투기 여부를 모르는 대다수의 무방비 구역 (진짜 깨끗한 구역과 신고 안 된 숨은 쓰레기 구역이 섞여 있음)

In [ ]:
# 1만 개의 격자 도화지 위에, 10개의 더미 피처(편의점 거리, 유동인구 등 가상 수치)를 찍어냄
X_dummy, y_true = make_classification(
    n_samples=10000, n_features=10, 
    n_informative=5, n_redundant=2, 
    weights=[0.9, 0.1], # 전체 중 90%는 U(Negative 아님), 10%는 P
    random_state=42
)

# 현실의 "분명 쓰레기가 산더미인데 민원 신고가 안 들어와서 공공데이터엔 없는 구역" 상황을 모사하기 위함.
# Positive 중 50%의 라벨을 억지로 0(Unlabeled)으로 훼손시켜버림
hidden_mask = (y_true == 1) & (np.random.rand(10000) > 0.5) 
y_pu = y_true.copy()
y_pu[hidden_mask] = 0

print(f"X (피처 행렬) Shape: {X_dummy.shape}")
print(f"실제 우주적 관점에서 쓰레기가 있는 구역: {sum(y_true==1)}개")
print(f"우리가 공공데이터에서 얻은 현실 라벨(PU): Positive {sum(y_pu==1)}개 | Unlabeled {sum(y_pu==0)}개")


## 2. PU-Bagging 앙상블 알고리즘 구현 (핵심 로직)
위에서 파악했듯 Unlabeled(0) 데이터에는 '숨겨진 진짜 투기 구역'이 섞여 있기 때문에, 단순히 0을 깨끗한 길로 취급하고 학습하면 모델 성능이 박살납니다.

이를 타파하기 위해 **모든 Positive** 데이터에, Unlabeled 그룹에서 Positive 개수만큼만 **조금씩 무작위로 추출하여 붙인 서브셋(Subset)**을 만들어 그 서브셋마다 작은 Random Forest를 각각 학습시킵니다. 
이 짓을 수십 번 반복하여 평균을 냄으로써 Unlabeled 속 Но이즈(Noise)를 압도하는 것이 PU-Bagging의 원리입니다.

In [ ]:
class PUBaggingRandomForest:
    def __init__(self, n_estimators=10, max_samples=None, random_state=42):
        self.n_estimators = n_estimators           # 서브셋을 뽑아서 학습할 총 횟수
        self.models = []                           # 안에 들어갈 RandomForest 용병들
        self.random_state = random_state
        
    def fit(self, X, y):
        # 1과 0 인덱스를 먼저 조사
        p_idx = np.where(y == 1)[0]
        u_idx = np.where(y == 0)[0]
        
        np.random.seed(self.random_state)
        
        for i in range(self.n_estimators):
            # Unlabeled 거대한 바다에서 Positive 인원수만큼만 랜덤하게 끌어옴 (Under-sampling)
            sampled_u_idx = np.random.choice(u_idx, size=len(p_idx), replace=True)
            
            # P와 랜덤 샘플 U를 합쳐서 이번 턴의 훈련소를 차림
            train_idx = np.concatenate([p_idx, sampled_u_idx])
            X_train = X[train_idx]
            # U는 임시로 0 (Negative) 취급해서 타겟을 만듦
            y_train = np.concatenate([np.ones(len(p_idx)), np.zeros(len(sampled_u_idx))])
            
            # 여기서 বে이스 분류기인 Random Forest 훈련
            rf = RandomForestClassifier(n_estimators=50, max_depth=5, n_jobs=-1, random_state=self.random_state+i)
            rf.fit(X_train, y_train)
            self.models.append(rf) # 훈련된 용병 적립
            
    def predict_proba(self, X):
        # 여러 의사결정나무 모델들이 예측한 확률값(Proba)들의 산술 평균 도출
        probas = np.array([model.predict_proba(X)[:, 1] for model in self.models])
        return probas.mean(axis=0)

print("✅ PU-Bagging 클래스 컴파일 완료!")

## 3. 베이스 모델 테스트 및 .gemini Rule 5 검증
객체지향적으로 작성한 우리의 커스텀 PU 모델을 Train/Test 분할 환경에 올려서, 과연 **가중치(Weight)가 0.0과 1.0 사이에서 정상적으로 사출되는지(Rule 5)** 엔진 충돌 방지 검증을 합니다.

In [ ]:
# 8:2로 나눔 (검증을 위해)
X_train, X_test, y_train, y_test_pu = train_test_split(X_dummy, y_pu, test_size=0.2, random_state=42)

# 모델 인스턴스화 (10번 배깅 앙상블)
pu_model = PUBaggingRandomForest(n_estimators=10)

print("🏋️‍♂️ PU-Learning 모델 학습 시작...")
pu_model.fit(X_train, y_train)
print("✅ 모델 학습 종료!")

# GraphHopper에 보낼 위험도 점수(Trash Score) 사출!
trash_scores = pu_model.predict_proba(X_test)

print("-" * 30)
print(f"📊 상위 10개 격자의 예측 점수 (Trash Score): \n{trash_scores[:10]}")
print(f"최소 점수: {trash_scores.min():.4f}")
print(f"최대 점수: {trash_scores.max():.4f}")
print(f"결측치(NaN) 렌더링 갯수: {np.isnan(trash_scores).sum()} 개")
print("-" * 30)

if (trash_scores.min() >= 0.0) and (trash_scores.max() <= 1.0) and not np.isnan(trash_scores).any():
    print("🟩 [PASS] Rule 5를 통과했습니다. GraphHopper에 이식해도 터지지 않는 안전한 텐서입니다.")
else:
    print("🟥 [FAIL] Rule 5를 위반했습니다. 점수 정규화 로직이 파괴되었습니다.")